In [ ]:
!pip install numpy sentence_transformers pandas faiss_cpu transformers

In [3]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer,CrossEncoder
from transformers import pipeline
from collections import defaultdict
import pandas as pd

In [4]:
Home_dir="100_movies"
CSV_FILE="100_movies"
JSON_FILE="100_movie_dialouge"

In [5]:
with open(f"{Home_dir}/{JSON_FILE}.json","r",encoding="utf-8") as file:
    data=json.load(file)

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "facebook/bart-large-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 2584.64it/s]


In [7]:
semantic_model=SentenceTransformer(
    "BAAI/bge-large-en-v1.5",
)

emotion_model = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None,
    device=0
)

intent_model=pipeline(
    "zero-shot-classification",
    model=model,
    tokenizer=tokenizer,
    device=0
)

reranker=CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2030.42it/s]


In [8]:
INTENT_LABELS = [

    "romantic confession",
    "love",
    "friendship",
    "sadness",
    "grief",
    "revenge",
    "motivation",
    "determination",
    "hope",
    "fear",
    "anger",
    "warning",
    "philosophical reflection",
    "identity crisis",
    "heroism",
    "betrayal",
    "justice",
    "loneliness",
    "self sacrifice",
    "inspiration"
]

In [ ]:
# import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# import numpy as np
# import torch

# MAX_TOKENS = 480

# # =========================================================
# # SAFE CHUNK FUNCTION
# # =========================================================

# def chunk_text(text):

#     # Handle None / NaN / empty text
#     if text is None:
#         return [""]

#     text = str(text).strip()

#     if len(text) == 0:
#         return [""]

#     tokens = tokenizer.encode(
#         text,
#         add_special_tokens=False
#     )

#     chunks = []

#     for i in range(0, len(tokens), MAX_TOKENS):

#         chunk_tokens = tokens[i:i + MAX_TOKENS]

#         # Skip empty chunks
#         if len(chunk_tokens) == 0:
#             continue

#         chunk = tokenizer.decode(
#             chunk_tokens,
#             skip_special_tokens=True,
#             clean_up_tokenization_spaces=True
#         )

#         chunks.append(chunk)

#     # Safety fallback
#     if len(chunks) == 0:
#         chunks = [""]

#     return chunks


# # =========================================================
# # STORAGE
# # =========================================================

# text_emotion_vectors = []

# # =========================================================
# # MAIN LOOP
# # =========================================================

# for i, item in enumerate(data):

#     try:

#         # =================================================
#         # GET TEXT
#         # =================================================

#         text = item.get("text", "")

#         # =================================================
#         # CHUNK TEXT
#         # =================================================

#         chunks = chunk_text(text)

#         # =================================================
#         # FILTER BAD CHUNKS
#         # =================================================

#         chunks = [
#             c for c in chunks
#             if isinstance(c, str) and len(c.strip()) > 0
#         ]

#         if len(chunks) == 0:
#             chunks = ["empty"]

#         # =================================================
#         # INFERENCE
#         # =================================================

#         with torch.no_grad():

#             chunk_outputs = emotion_model(
#                 chunks,
#                 truncation=True,
#                 padding=True,
#                 batch_size=8
#             )

#         # =================================================
#         # EXTRACT SCORES
#         # =================================================

#         chunk_score_vectors = []

#         for output in chunk_outputs:

#             # Safety check
#             if not isinstance(output, list):
#                 continue

#             scores = []

#             for x in output:

#                 score = float(x["score"])

#                 # NaN protection
#                 if np.isnan(score):
#                     score = 0.0

#                 scores.append(score)

#             if len(scores) > 0:
#                 chunk_score_vectors.append(scores)

#         # =================================================
#         # HANDLE EMPTY OUTPUT
#         # =================================================

#         if len(chunk_score_vectors) == 0:

#             # fallback zero vector
#             num_emotions = len(chunk_outputs[0]) \
#                 if len(chunk_outputs) > 0 else 1

#             final_scores = np.zeros(
#                 num_emotions,
#                 dtype=np.float32
#             )

#         else:

#             # =================================================
#             # AVERAGE SCORES
#             # =================================================

#             final_scores = np.mean(
#                 chunk_score_vectors,
#                 axis=0
#             ).astype(np.float32)

#         text_emotion_vectors.append(final_scores)

#         # =================================================
#         # LOGGING
#         # =================================================

#         if i % 100 == 0:

#             print(
#                 f"Processed: {i}/{len(data)}"
#             )

#     except Exception as e:

#         print(f"\nERROR at index {i}")
#         print(e)

#         # Add fallback vector
#         text_emotion_vectors.append(
#             np.zeros(6, dtype=np.float32)
#         )

#         # IMPORTANT:
#         # CUDA errors poison runtime sometimes
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()

# # =========================================================
# # FINAL ARRAY
# # =========================================================

# text_emotion_vectors = np.array(
#     text_emotion_vectors,
#     dtype=np.float32
# )

# # =========================================================
# # SAVE
# # =========================================================

# np.save(
#     f"{Home_dir}/embeddings/text_emotion.npy",
#     text_emotion_vectors
# )

# print("\nFinal Shape:")
# print(text_emotion_vectors.shape)

In [ ]:
ids=[]
summary_semantic_vectors = []
summary_emotion_vectors = []
summary_intent_vectors = []

text_semantic_vectors = []
text_emotion_vectors = []
text_intent_vectors = []


# SUMMARY SEMANTIC
# for i, item in enumerate(data):
#     summary = item["summary"]

#     summary_semantic = semantic_model.encode(summary,normalize_embeddings=True)
#     summary_semantic_vectors.append(summary_semantic)

#     if i % 100 == 0:
#         print(f"SUMMARY SEMANTIC Processed: {i}")

# summary_semantic_vectors=np.array(summary_semantic_vectors,dtype=np.float32)
# np.save(f"{Home_dir}/embeddings/summary_semantic.npy",summary_semantic_vectors)
# summary_semantic_vectors=np.array([])

# # SUMMARY EMOTION
# for i, item in enumerate(data):
#     summary = item["summary"]
# #
#     summary_emotion_result = emotion_model(summary)[0]
#     summary_emotion_vector = [x["score"]for x in summary_emotion_result]
#     summary_emotion_vectors.append(summary_emotion_vector)
#     if i % 100 == 0:
#         print(f"SUMMARY EMOTION Processed: {i}")

# summary_emotion_vectors=np.array(summary_emotion_vectors,dtype=np.float32)
# np.save(f"{Home_dir}/embeddings/summary_emotion.npy",summary_emotion_vectors)
# summary_emotion_vectors=np.array([])

# # SUMMARY INTENT
# for i, item in enumerate(data):
#     summary = item["summary"]
  
#     summary_intent_result = intent_model(summary,INTENT_LABELS)
#     summary_intent_vector = (summary_intent_result["scores"])
#     summary_intent_vectors.append(summary_intent_vector)

#     if i % 100 == 0:
#         print(f"SUMMARY INTENT Processed: {i}")

# summary_intent_vectors=np.array(summary_intent_vectors,dtype=np.float32)
# np.save(f"{Home_dir}/embeddings/summary_intent.npy",summary_intent_vectors)
# summary_intent_vectors=np.array([])


# # TEXT INTENT
# for i, item in enumerate(data):
#     text = item["text"]

#     text_intent_result = intent_model(text,INTENT_LABELS)
#     text_intent_vector = (text_intent_result["scores"])
#     text_intent_vectors.append(text_intent_vector)

#     if i % 100 == 0:
#         print(f"TEXT INTENT Processed: {i}")

# text_intent_vectors=np.array(text_intent_vectors,dtype=np.float32)
# np.save(f"{Home_dir}/embeddings/text_intent.npy",text_intent_vectors)
# text_intent_vectors=np.array([])


# TEXT EMOTION
# for i, item in enumerate(data):
#     text = item["text"]

#     text_emotion_result = emotion_model(text)[0]
#     text_emotion_vector = [x["score"] for x in text_emotion_result]
#     text_emotion_vectors.append(text_emotion_vector)

#     if i % 100 == 0:
#         print(f"TEXT EMOTION Processed: {i}")

# text_emotion_vectors=np.array(text_emotion_vectors,dtype=np.float32)
# np.save(f"{Home_dir}/embeddings/text_emotion.npy",text_emotion_vectors)
# text_emotion_vectors=np.array([])

# TEXT SEMANTIC
for i, item in enumerate(data):
    text = item["text"]

    text_semantic = semantic_model.encode(text,normalize_embeddings=True)
    text_semantic_vectors.append(text_semantic)
    if i % 100 == 0:
        print(f"TEXT SEMANTIC Processed: {i}")

text_semantic_vectors=np.array(text_semantic_vectors,dtype=np.float32)
np.save(f"{Home_dir}/embeddings/text_semantic.npy",text_semantic_vectors)
text_semantic_vectors=np.array([])


print("\nFinished Embedding Creation")

In [9]:
import numpy as np
import os

# Path to your .npy file
file_path = f"{Home_dir}/embeddings/summary_intent.npy"

# File size on disk
size_bytes = os.path.getsize(file_path)
size_mb = size_bytes / (1024 ** 2)

print(f"File size: {size_bytes} bytes")
print(f"File size: {size_mb:.2f} MB")

# Load array and inspect
arr = np.load(file_path)

print(f"Shape: {arr.shape}")
print(f"Data type: {arr.dtype}")
print(f"Array size (elements): {arr.size}")

# Memory occupied by array
print(f"Memory used by array: {arr.nbytes / (1024 ** 2):.2f} MB")

File size: 590688 bytes
File size: 0.56 MB
Shape: (7382, 20)
Data type: float32
Array size (elements): 147640
Memory used by array: 0.56 MB


In [ ]:
res=faiss.StandardGpuResources()


def save_all_indices():
    mappings = {
        f"{Home_dir}/embeddings/summary_semantic.npy": f"{Home_dir}/index/summary_semantic.index",
        f"{Home_dir}/embeddings/summary_emotion.npy": f"{Home_dir}/index/summary_emotion.index",
        f"{Home_dir}/embeddings/summary_intent.npy": f"{Home_dir}/index/summary_intent.index",
        f"{Home_dir}/embeddings/text_semantic.npy": f"{Home_dir}/index/text_semantic.index",
        f"{Home_dir}/embeddings/text_emotion.npy": f"{Home_dir}/index/text_emotion.index",
        f"{Home_dir}/embeddings/text_intent.npy": f"{Home_dir}/index/text_intent.index"
    }
    
    for npy_path, index_path in mappings.items():
        vectors = np.load(npy_path)
        dim = vectors.shape[1]
        cpu_index = faiss.IndexFlatIP(dim)
        cpu_index.add(vectors)
        faiss.write_index(cpu_index, index_path)


In [10]:
def load_gpu_index(index_path):
    cpu_index = faiss.read_index(index_path)
    return cpu_index

In [ ]:
save_all_indices()

In [14]:
summary_semantic_index = load_gpu_index(f"{Home_dir}/index/summary_semantic.index")
summary_emotion_index = load_gpu_index(f"{Home_dir}/index/summary_emotion.index")
summary_intent_index = load_gpu_index(f"{Home_dir}/index/summary_intent.index")
text_semantic_index = load_gpu_index(f"{Home_dir}/index/text_semantic.index")
text_emotion_index = load_gpu_index(f"{Home_dir}/index/text_emotion.index")
text_intent_index = load_gpu_index(f"{Home_dir}/index/text_intent.index")

In [ ]:
def search_dialogue(query, top_k=10):
    print("\n")
    print("=" * 100)
    print("QUERY:", query)
    print("=" * 100)

    query_semantic = semantic_model.encode(query, normalize_embeddings=True).astype(np.float32)


    all_results = defaultdict(float)
    retrieval_scores = {}

    scores, ids_found = summary_semantic_index.search(
        query_semantic.reshape(1, -1), top_k
    )
    for score, idx in zip(scores[0], ids_found[0]):
        weighted_score = 0.25 * float(score)
        all_results[idx] += weighted_score

    scores, ids_found = text_semantic_index.search(query_semantic.reshape(1, -1), top_k)
    for score, idx in zip(scores[0], ids_found[0]):
        weighted_score = 0.35 * float(score)
        all_results[idx] += weighted_score


    candidates = sorted(all_results.items(), key=lambda x: x[1], reverse=True)[:top_k]

    for idx, score in candidates:
        retrieval_scores[idx] = score

    summary_pairs = []
    text_pairs = []
    candidate_ids = []
    for idx, score in candidates:
        item = data[idx]
        summary_pairs.append((query, item["summary"]))
        text_pairs.append((query, item["text"]))
        candidate_ids.append(idx)

    summary_rerank_scores = reranker.predict(summary_pairs)
    text_rerank_scores = reranker.predict(text_pairs)

    final_results = []
    for idx, summary_score, text_score in zip(candidate_ids, summary_rerank_scores, text_rerank_scores):
        retrieval_score = retrieval_scores[idx]
        final_score = (0.30 * retrieval_score + 0.30 * float(summary_score) + 0.40 * float(text_score))
        final_results.append((idx, final_score, retrieval_score, float(summary_score), float(text_score)))

    final_results.sort(key=lambda x: x[1], reverse=True)

    csv_file=pd.read_csv(f"{Home_dir}/{CSV_FILE}.csv")
    print("\n")
    print("=" * 100)
    print("FINAL RESULTS")
    print("=" * 100)
    for rank, result in enumerate(final_results):
        idx = result[0]
        final_score = result[1]
        retrieval_score = result[2]
        summary_score = result[3]
        text_score = result[4]
        item = data[idx]
        # print("\n")
        # print("-" * 100)
        # print(f"RANK: {rank + 1}")
        print(f"FINAL SCORE: {final_score:.4f}")
        # print(f"RETRIEVAL SCORE: {retrieval_score:.4f}")
        # print(f"SUMMARY RERANK: {summary_score:.4f}")
        # print(f"TEXT RERANK: {text_score:.4f}")
        # print(f"ID: {item['_id']}")
        result = csv_file.loc[csv_file['imdb_id'] == item['_id'].split('_')[0],'original_title']
        movie_name = result.iloc[0] if not result.empty else "Not Found"
        print(f"MOVIE_NAME: {movie_name}")
        # print("\nSUMMARY:")
        # print(item["summary"])
        # print("\nTEXT:")
        # print(item["text"])
        # print("-" * 100)

In [20]:
def search_dialogue_all(query, top_k=10):
    print("\n")
    print("=" * 100)
    print("QUERY:", query)
    print("=" * 100)

    query_semantic = semantic_model.encode(query, normalize_embeddings=True).astype(
        np.float32
    )

    query_emotion_result = emotion_model(query)[0]
    query_emotion = np.array(
        [x["score"] for x in query_emotion_result], dtype=np.float32
    )

    query_intent_result = intent_model(query, INTENT_LABELS)
    query_intent = np.array(query_intent_result["scores"], dtype=np.float32)

    all_results = defaultdict(float)
    retrieval_scores = {}

    scores, ids_found = summary_semantic_index.search(
        query_semantic.reshape(1, -1), top_k
    )
    for score, idx in zip(scores[0], ids_found[0]):
        weighted_score = 0.25 * float(score)
        all_results[idx] += weighted_score

    scores, ids_found = summary_emotion_index.search(query_emotion.reshape(1, -1), top_k)
    for score, idx in zip(scores[0], ids_found[0]):
        weighted_score = 0.10 * float(score)
        all_results[idx] += weighted_score

    scores, ids_found = summary_intent_index.search(query_intent.reshape(1, -1), top_k)
    for score, idx in zip(scores[0], ids_found[0]):
        weighted_score = 0.10 * float(score)
        all_results[idx] += weighted_score

    scores, ids_found = text_semantic_index.search(query_semantic.reshape(1, -1), top_k)
    for score, idx in zip(scores[0], ids_found[0]):
        weighted_score = 0.35 * float(score)
        all_results[idx] += weighted_score

    scores, ids_found = text_emotion_index.search(query_emotion.reshape(1, -1), top_k)
    for score, idx in zip(scores[0], ids_found[0]):
        weighted_score = 0.15 * float(score)
        all_results[idx] += weighted_score

    scores, ids_found = text_intent_index.search(query_intent.reshape(1, -1), top_k)
    for score, idx in zip(scores[0], ids_found[0]):
        weighted_score = 0.05 * float(score)
        all_results[idx] += weighted_score

    candidates = sorted(all_results.items(), key=lambda x: x[1], reverse=True)[:top_k]

    for idx, score in candidates:
        retrieval_scores[idx] = score

    summary_pairs = []
    text_pairs = []
    candidate_ids = []
    for idx, score in candidates:
        item = data[idx]
        summary_pairs.append((query, item["summary"]))
        text_pairs.append((query, item["text"]))
        candidate_ids.append(idx)

    summary_rerank_scores = reranker.predict(summary_pairs)
    text_rerank_scores = reranker.predict(text_pairs)

    final_results = []
    for idx, summary_score, text_score in zip(
        candidate_ids, summary_rerank_scores, text_rerank_scores
    ):
        retrieval_score = retrieval_scores[idx]
        final_score = (
            0.30 * retrieval_score
            + 0.30 * float(summary_score)
            + 0.40 * float(text_score)
        )
        final_results.append(
            (idx, final_score, retrieval_score, float(summary_score), float(text_score))
        )

    final_results.sort(key=lambda x: x[1], reverse=True)

    csv_file = pd.read_csv(f"{Home_dir}/{CSV_FILE}.csv")
    print("\n")
    print("=" * 100)
    print("FINAL RESULTS")
    print("=" * 100)
    for rank, result in enumerate(final_results):
        idx = result[0]
        final_score = result[1]
        retrieval_score = result[2]
        summary_score = result[3]
        text_score = result[4]
        item = data[idx]
        # print("\n")
        # print("-" * 100)
        # print(f"RANK: {rank + 1}")
        print(f"FINAL SCORE: {final_score:.4f}")
        # print(f"RETRIEVAL SCORE: {retrieval_score:.4f}")
        # print(f"SUMMARY RERANK: {summary_score:.4f}")
        # print(f"TEXT RERANK: {text_score:.4f}")
        # print(f"ID: {item['_id']}")
        result = csv_file.loc[
            csv_file["imdb_id"] == item["_id"].split("_")[0], "original_title"
        ]
        movie_name = result.iloc[0] if not result.empty else "Not Found"
        print(f"MOVIE_NAME: {movie_name}")
        # print("\nSUMMARY:")
        # print(item["summary"])
        # print("\nTEXT:")
        # print(item["text"])
        # print("-" * 100)

In [22]:
search_dialogue(
    "your mother respect me ,you have to  like her. if you move you will die there is an posion stick near it"
)



QUERY: your mother respect me ,you have to  like her. if you move you will die there is an posion stick near it


FINAL RESULTS
FINAL SCORE: -4.8575
MOVIE_NAME: Dune: Part One
FINAL SCORE: -5.2024
MOVIE_NAME: Avengers: Endgame
FINAL SCORE: -5.3094
MOVIE_NAME: How to Train Your Dragon
FINAL SCORE: -5.6958
MOVIE_NAME: Dune: Part One
FINAL SCORE: -5.9362
MOVIE_NAME: No Country for Old Men
FINAL SCORE: -6.4152
MOVIE_NAME: The Incredibles
FINAL SCORE: -6.4744
MOVIE_NAME: Requiem for a Dream
FINAL SCORE: -7.0570
MOVIE_NAME: Joker
FINAL SCORE: -7.1375
MOVIE_NAME: Avatar
FINAL SCORE: -7.4602
MOVIE_NAME: Black Panther


In [23]:
search_dialogue_all(
    "your mother respect me ,you have to  like her. if you move you will die there is an posion stick near it"
)



QUERY: your mother respect me ,you have to  like her. if you move you will die there is an posion stick near it


FINAL RESULTS
FINAL SCORE: -4.8575
MOVIE_NAME: Dune: Part One
FINAL SCORE: -5.2024
MOVIE_NAME: Avengers: Endgame
FINAL SCORE: -5.3094
MOVIE_NAME: How to Train Your Dragon
FINAL SCORE: -5.6958
MOVIE_NAME: Dune: Part One
FINAL SCORE: -5.9362
MOVIE_NAME: No Country for Old Men
FINAL SCORE: -6.4152
MOVIE_NAME: The Incredibles
FINAL SCORE: -6.4744
MOVIE_NAME: Requiem for a Dream
FINAL SCORE: -7.0570
MOVIE_NAME: Joker
FINAL SCORE: -7.1375
MOVIE_NAME: Avatar
FINAL SCORE: -7.4602
MOVIE_NAME: Black Panther
